In [2]:
import sys

ruta_raiz = 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/'
# ruta_raiz = '/Users/antonio_lopez_torres/Documents/GitHub/RRAM_Simulation/' # Ruta en el mac
sys.path.append(ruta_raiz)

from RRAM import Plot_PostProcess as pplt
from RRAM import Constants as cte

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
with open("Datos_Experimentales\Prim_Set.txt", "r") as file:
    medidos = [float(x) for line in file for x in line.strip().split()]
    
print(medidos)

[0.01, 1.859e-05, 0.0655635, 0.02, 3.6825e-05, 0.1131296, 0.03, 5.5115e-05, 0.1615196, 0.04, 7.3705e-05, 0.2091181, 0.05, 9.2535e-05, 0.2573504, 0.06, 0.00011125, 0.3000123, 0.07, 0.00013025, 0.3475375, 0.08, 0.0001505, 0.3955972, 0.09, 0.00017015, 0.4434104, 0.1, 0.00019085, 0.4914431, 0.11, 0.0002129, 0.5394187, 0.12, 0.0002317, 0.5873338, 0.13, 0.000255, 0.6351639, 0.14, 0.00027645, 0.6842107, 0.15, 0.0003022, 0.7326715, 0.16, 0.0003268, 0.7794428, 0.17, 0.00035075, 0.8284682, 0.18, 0.00038205, 0.8758993, 0.19, 0.0004135, 0.9246412, 0.2, 0.00043465, 0.9724637, 0.21, 0.00045535, 1.020476, 0.22, 0.0004848, 1.068564, 0.23, 0.00050835, 1.116537, 0.24, 0.0005295, 1.164238, 0.25, 0.00055175, 1.212499, 0.26, 0.0005851, 1.260484, 0.27, 0.00060815, 1.308507, 0.28, 0.00064255, 1.35644, 0.29, 0.0006825, 1.403887, 0.3, 0.0007122, 1.451091, 0.31, 0.00075535, 1.499447, 0.32, 0.00080705, 1.547473, 0.33, 0.0008342, 1.59546, 0.34, 0.00085765, 1.644297, 0.35, 0.0008985, 1.692291, 0.36, 0.00092775, 1.

In [ ]:
def leer_intensidades_en_intervalo(v_min, v_max):
    nombre_archivo = "Datos_Experimentales/Ciclos_Experimentales/Cycle_p_1000.txt"

    # Cargar solo las dos primeras columnas con NumPy
    datos = np.loadtxt(nombre_archivo, usecols=(0, 1))  # Columna 0 = voltaje, Columna 1 = intensidad

    voltajes = datos[:, 0]  # Extraer columna de voltaje
    intensidades = datos[:, 1]  # Extraer columna de intensidad

    # Filtrar intensidades donde el voltaje esté en el intervalo [v_min, v_max]
    intensidades_filtradas = intensidades[(voltajes >= v_min) & (voltajes <= v_max)]

    # Calcular la media de las intensidades filtradas
    # Evita error si no hay datos
    i_media = np.mean(intensidades_filtradas) if intensidades_filtradas.size > 0 else np.nan
    
    if np.isnan(i_media):
        print("No hay datos en el intervalo [", v_min, ",", v_max, "]")
    
    return i_media


# Ejemplo de uso
V_div = 0.063282
v_min = 0.5

intensidades_medias = np.array([leer_intensidades_en_intervalo(
    v_min + i * V_div, v_min + (i + 1) * V_div) for i in range(10)])

print("Vector de intensidades medias:", intensidades_medias)

i_sim = np.array([0.009, 0.0109, 0.015, 0.0139, 0.0169, 0.0189, 0.0199, 0.0229, 0.0239, 0.0259])

# divido I_sim entre i_media de forma entera y redonde al entero más cercano
i_sim = np.round(i_sim / intensidades_medias)

print("Vector de intensidades simuladas:", i_sim)

Vector de intensidades medias: [0.00780871 0.01030138 0.012155   0.01396464 0.01615708 0.01819333
 0.01999929 0.02211333 0.02383958 0.02524429]
Vector de intensidades simuladas: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Configuraciones de estilo para todos los plots

In [5]:
# region Configuracion del plot
def config_ax(ax):
    ax.grid(which='major', color='#DDDDDD', linewidth=0.8, zorder=-1)
    # Show the minor grid as well. Style it in very light gray as a thin,
    # dotted line.
    ax.grid(which='minor', color='#DEDEDE', linestyle=':', linewidth=0.5, zorder=-1)
    # Make the minor ticks and gridlines show.
    ax.minorticks_on()

    ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)


def setup_plt(plt, latex=True, scaling=1):
    plt.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": latex,
            "font.family": "fourier",
            "text.latex.preamble": "\n".join([  # plots will use this preamble
                r"\usepackage[utf8]{inputenc}",
                r"\usepackage[T1]{fontenc}",
                r"\usepackage{siunitx}",
            ])
        }
    )

    SMALL_SIZE = 8 * scaling
    MEDIUM_SIZE = 10 * scaling
    BIGGER_SIZE = 12 * scaling

    plt.rc('font', size=SMALL_SIZE)  # controls default text sizes
    plt.rc('axes', titlesize=SMALL_SIZE)  # fontsize of the axes title
    plt.rc('axes', labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
    plt.rc('xtick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
    plt.rc('ytick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
    plt.rc('legend', fontsize=SMALL_SIZE)  # legend fontsize
    plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title


setup_plt(plt, latex=True, scaling=1.5)

In [6]:
carpeta_results = 'Results'

ruta_raiz = 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/'

simulation_path = os.path.join(carpeta_results, f'simulation_{num_simulation + 1}/')
figures_path = os.path.join(carpeta_results, 'Figures')


set_simulation_path = os.path.join(carpeta_results, f'simulation_{num_simulation + 1}/set/')
os.makedirs(set_simulation_path, exist_ok=True)

reset_simulation_path = os.path.join(carpeta_results, f'simulation_{num_simulation + 1}/reset/')
os.makedirs(reset_simulation_path, exist_ok=True)

NameError: name 'os' is not defined

In [ ]:
# region Unir todos los datos en un solo archivo csv TODO: necesita revision cuando se unen los datos
df_pset = pd.read_csv(set_simulation_path + f'Resultados_pp_set_{num_simulation +1}.csv')
df_sset = pd.read_csv(set_simulation_path + f'Resultados_sp_set_{num_simulation +1}.csv')
df_preset = pd.read_csv(reset_simulation_path + f'resultados_pp_reset_{num_simulation +1}.csv')
df_sreset = pd.read_csv(reset_simulation_path + f'resultados_sp_reset_{num_simulation +1}.csv')

# Concatenar los DataFrames sin duplicar el encabezado
data_frame_simulation = pd.concat([df_pset, df_sset, df_preset, df_sreset])

# Guardar el DataFrame combinado en un archivo CSV
data_frame_simulation.to_csv(f'Results/Datos_simulacion_completa_{num_simulation +1}.csv', index=False)
print("Todos los archivos CSV se han combinado y guardado exitosamente.")
# endregion

# region Representar datos
data_path_pp_set = set_simulation_path + f'resultados_pp_set_{num_simulation +1}.csv'
data_path_sp_set = set_simulation_path + f'resultados_sp_set_{num_simulation +1}.csv'
data_path_pp_reset = reset_simulation_path + f'resultados_pp_reset_{num_simulation +1}.csv'
data_path_sp_reset = reset_simulation_path + f'resultados_sp_reset_{num_simulation +1}.csv'

df_pset = pd.read_csv(data_path_pp_set, dtype=float)
df_sset = pd.read_csv(data_path_sp_set, dtype=float)
df_preset = pd.read_csv(data_path_pp_reset, dtype=float)
df_sreset = pd.read_csv(data_path_sp_reset, dtype=float)

global_tittle = 'Intensidad vs Voltaje'

save_path = simulation_path + f'Figures/Intensidad_Voltaje_simulation_{num_simulation+1}'

i_ps = np.array(df_pset['Intensidad [A]'])
i_ss = np.array(df_sset['Intensidad [A]'])
i_pr = np.array(df_preset['Intensidad [A]'])
i_sr = np.array(df_sreset['Intensidad [A]'])
v_ps = np.array(df_pset['Voltaje [V]'])
v_ss = np.array(df_sset['Voltaje [V]'])
v_pr = np.array(df_preset['Voltaje [V]'])
v_sr = np.array(df_sreset['Voltaje [V]'])

fig, axes = plt.subplots()

pplt.config_ax(axes)

axes.set_xlabel('Voltaje [V]')
axes.set_ylabel('Intensidad [A]')
axes.set_yscale('log')

axes.set_title(global_tittle, fontsize=18, pad=15)

axes.scatter(v_ps, i_ps, color='blue', s=0.2, label='Primera parte Set')
axes.scatter(v_ss, i_ss, color='red', s=0.2, label='Segunda parte Set')
axes.scatter(v_pr, i_pr, color='green', s=0.2, label='Primera parte Reset')
axes.scatter(v_sr, i_sr, color='pink', s=0.2, label='Segunda parte Reset')

plt.legend()

# Ruta proporcionada
ruta_exp_data = ruta_raiz + 'Datos_Experimentales/Ciclos_Experimentales'
ruta_archivo_set = ruta_exp_data + '/Cycle_p_1000.txt'
ruta_archivo_reset = ruta_exp_data + '/Cycle_n_1000.txt'

# Leer datos del archivo
data_set = np.loadtxt(ruta_archivo_set)
data_reset = np.loadtxt(ruta_archivo_reset)
# Asumimos que los datos están en dos columnas: x e y

x_set = data_set[:, 0]
y_set = data_set[:, 1]

# Data reset
x_reset = data_reset[:, 0]
y_reset = abs(data_reset[:, 1])

# Añado los valores de las variables que estoy cambiando, para eso tengo q ver dentro de la carpeta de init_data el nombre de cada documento, si el nombre conindice con una variable del diccionario, añado el valor que está tomando en la simulación en la representación

# Leer los valores de las variables desde los archivos en Initial_data
init_data_path = 'Initial_data'
variables = {}

for filename in os.listdir(init_data_path):
    if filename.endswith('.pkl'):
        variable_name = filename.split('.')[0]
        with open(os.path.join(init_data_path, filename), 'rb') as f:
            variables[variable_name] = pickle.load(f)

# Crear el subtítulo con los valores de las variables
espacios = ' ' * 3  # Define el número de espacios que deseas añadir

# Construye la lista de subtítulos con espacios y saltos de línea cada tres valores
subtitles = []
for i, (variable_name, value) in enumerate(variables.items()):
    subtitles.append(f'{variable_name} = {value[num_simulation]}')
    if (i + 1) % 3 == 0:
        subtitles.append('\n')
    else:
        subtitles.append(espacios)

# Une los subtítulos en una sola cadena
subtitle = ''.join(subtitles).strip()
fig.suptitle(subtitle, fontsize=11, y=1.05)  # Ajusta el valor de y según sea necesario

# Crear la gráfica scatter
axes.plot(x_set, y_set, 'black', label='Set experimental')
axes.plot(x_reset, y_reset, 'black', label='Reset experimental')

# fig.savefig(save_path + '.pdf', bbox_inches='tight')
# fig.savefig(figures_path + f'/Intensidad_Voltaje_simulation_{num_simulation + 1}.pdf', bbox_inches='tight')
fig.savefig(figures_path + f'/Intensidad_Voltaje_simulation_{num_simulation + 1}.png', bbox_inches='tight')